# XMCA-Net-Lite: 00 - Environment & Data Preparation
### Purpose
Download and standardize the four evaluation datasets (POPE, FaithScore/LLaVA-1k, A-OKVQA, InfoSeek) with strict 2,000 maximum sample caps. Implement reproducible image-grouped 70/10/20 train/val/test splits to guarantee zero data leakage between splits.
**Research Claim Limit**: Reports proxy attribution only; no causal claims.


In [ ]:
# Environment Setup & Dependency Installation (Kaggle T4)
import sys, os
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("/kaggle/working/xmca") if Path("/kaggle/working").exists() else Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Working Directory: {PROJECT_ROOT}")


In [ ]:
from src.config import load_config, set_seed
from src.data.loader import load_dataset_records
from src.data.splitter import split_by_image_id, save_and_manifest_splits
from src.utils.logging_metadata import save_run_metadata

config = load_config()
seed = config["project"]["default_seed"]
set_seed(seed)
print("Configuration loaded. Base directory:", config["paths"]["base_dir"])


In [ ]:
# Download and process the four datasets with strict 2,000 caps & image-grouped splits
splits_dir = Path(config["paths"]["splits_dir"])
dataset_configs = config["datasets"]["sources"]
split_manifests = {}

for ds_key, ds_meta in dataset_configs.items():
    print(f"\n--- Processing {ds_meta['name']} (Max: {ds_meta['max_samples']}) ---")
    records = load_dataset_records(
        dataset_name=ds_key,
        max_samples=ds_meta["max_samples"],
        seed=seed
    )
    print(f"Loaded {len(records)} candidate samples.")
    
    train, val, test = split_by_image_id(
        records=records,
        train_ratio=config["datasets"]["split_ratios"]["train"],
        val_ratio=config["datasets"]["split_ratios"]["val"],
        test_ratio=config["datasets"]["split_ratios"]["test"],
        seed=seed,
        max_total_samples=ds_meta["max_samples"]
    )
    
    manifest = save_and_manifest_splits(ds_key, train, val, test, splits_dir)
    split_manifests[ds_key] = manifest
    print(f"Splits saved -> Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

# Save run metadata and configuration copy permanently
save_run_metadata(splits_dir, config, seed, "00_environment_data", {"manifests": split_manifests})
print("\nStep 00 completed successfully. All split manifests permanently recorded.")
